# 38 — Does an angle-selected pool compose better?

Notebook 37 showed that re-selecting the bf16 pool by the bounded d_cos
metric replaces 15/40 concepts — the two metrics agree globally
(Spearman 0.69 over all candidates) but disagree in the top tail where
selection happens. This notebook answers the follow-up: **does the
membership difference propagate to composition results?**

Design: the notebook-33 group-selection algorithm (10 ρ-spanning pairs,
usage cap 2; 30 nested triples at distant/mid/close entanglement) applied
to the **d_cos pool**, then the three plain families (blend / vote /
schedule) at the standard settings (bf16 Llama, k=4, steps=24, both
metric tracks). Comparison target: notebook 33's rows for the same three
methods on the expression pool. Pools imply different groups, so this is
a pool-level comparison — the design question is precisely whether the
selection metric changes downstream composition quality; group mean-ρ
distributions are reported alongside to check comparability.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

from frames.evaluation.harness import EvaluationHarness
from frames.linalg.frame import Frame
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
N_BASE_PAIRS = 10
MAX_USES_PER_CONCEPT = 2

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/37_pool_dcos.json").read_text())["pool_dcos"]
print(f"d_cos pool: {len(POOL)} concepts")


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/38_scaling_dcos_pool.jsonl")
GROUPS_PATH = Path("resources/38_groups_dcos.json")
LOG_PATH.unlink(missing_ok=True)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
concept = {}
for name in POOL:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
names = sorted(concept)
kmax = max(len(concept[n]) for n in names)
stack = torch.cat(
    [F.pad(concept[n].tensor, (0, kmax - len(concept[n]))) for n in names]
)
RHO = Frame.correlation(stack, stack).float().cpu()


def rho(a: str, b: str) -> float:
    return float(RHO[names.index(a), names.index(b)])


all_pairs = sorted(
    (
        {"a": a, "b": b, "pair_rho": rho(a, b)}
        for i, a in enumerate(names)
        for b in names[i + 1 :]
    ),
    key=lambda p: p["pair_rho"],
)
targets = [
    all_pairs[round(i * (len(all_pairs) - 1) / (N_BASE_PAIRS - 1))]["pair_rho"]
    for i in range(N_BASE_PAIRS)
]
PAIRS, uses = [], {}
for t in targets:
    for p in sorted(all_pairs, key=lambda p: abs(p["pair_rho"] - t)):
        key = (p["a"], p["b"])
        if key in {(q["a"], q["b"]) for q in PAIRS}:
            continue
        if uses.get(p["a"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        if uses.get(p["b"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        PAIRS.append(p)
        uses[p["a"]] = uses.get(p["a"], 0) + 1
        uses[p["b"]] = uses.get(p["b"], 0) + 1
        break
if len(PAIRS) < N_BASE_PAIRS:
    raise RuntimeError("usage cap prevented an even pair spread")

TRIPLES = []
for p in PAIRS:
    a, b = p["a"], p["b"]
    ranked = sorted(
        (c for c in names if c not in (a, b)),
        key=lambda c: max(rho(c, a), rho(c, b)),
    )
    picks = {"distant": ranked[0], "mid": ranked[len(ranked) // 2], "close": ranked[-1]}
    for add_regime, c in picks.items():
        TRIPLES.append(
            {
                "a": a,
                "b": b,
                "c": c,
                "add_regime": add_regime,
                "add_rho": max(rho(c, a), rho(c, b)),
                "mean_rho": (rho(a, b) + rho(a, c) + rho(b, c)) / 3,
                "pair_rho": p["pair_rho"],
            }
        )

GROUPS_PATH.write_text(
    json.dumps(
        {
            "config": {"pool": "resources/37_pool_dcos.json (pool_dcos)"},
            "pairs": PAIRS,
            "triples": TRIPLES,
        },
        indent=1,
    )
)
print(
    f"{len(PAIRS)} pairs (rho {PAIRS[0]['pair_rho']:.2f}.."
    f"{PAIRS[-1]['pair_rho']:.2f}), {len(TRIPLES)} triples"
)

In [ ]:
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

for group in [*PAIRS, *TRIPLES]:
    synsets = [s for s in (group["a"], group["b"], group.get("c")) if s]
    cs = [concept[s] for s in synsets]
    methods = {
        "blend": lambda: fur.generate_with_topk_guide(
            prompts, guide=Concept.average(cs), k=K, steps=STEPS
        ),
        "vote": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, normalize="zscore"
        ),
        "schedule": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, scorer=RoundRobin()
        ),
    }
    for method, generate in methods.items():
        torch.cuda.empty_cache()
        texts, probe = generate()
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": method,
                "n": len(synsets),
                "pool": "dcos",
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    print(f"n={len(synsets)} {' + '.join(s.split('.')[0] for s in synsets)}: done")

In [ ]:
def rows_from(path, methods, pool_label):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c")) if s]
        deltas = [r["expression"][s]["delta"] for s in synsets]
        cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
        rows.append(
            {
                "pool": pool_label,
                "method": c["method"],
                "n": c["n"],
                "joint": min(deltas),
                "joint_cos": min(cosines),
                "all_positive": min(deltas) > 0,
                "all_positive_cos": min(cosines) > 0,
                "mean_rho": c.get("mean_rho", c["pair_rho"]),
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


plain = {"blend", "vote", "schedule"}
dcos = rows_from(LOG_PATH, plain, "dcos")
expr = rows_from("resources/33_scaling3_bf16.jsonl", plain, "expr")
results = pd.concat([dcos, expr], ignore_index=True)
print(
    "group mean-rho by pool:",
    results.groupby("pool")["mean_rho"].mean().round(3).to_dict(),
)

summary = (
    results.groupby(["pool", "method", "n"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive=("all_positive", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/38_pool_comparison.csv", index=False)
print(summary.round(4).to_string(index=False))

In [ ]:
order = ["blend", "vote", "schedule"]
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for ax, (metric, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share ALL concepts improved (cosine)"),
        ("joint", "mean joint expression gain"),
    ),
):
    x = 0
    ticks, ticklabels = [], []
    for n in (2, 3):
        for method in order:
            for pool, color in (("expr", "tab:blue"), ("dcos", "tab:red")):
                sub = summary[
                    (summary["pool"] == pool)
                    & (summary["method"] == method)
                    & (summary["n"] == n)
                ]
                ax.bar(x, sub[metric].iloc[0], width=0.85, color=color)
                x += 1
            ticks.append(x - 1.5)
            ticklabels.append(f"{method}\nn={n}")
            x += 0.6
        x += 1
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticklabels, fontsize=7)
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[0].legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="tab:blue", label="expression pool (33)"),
        plt.Rectangle((0, 0), 1, 1, color="tab:red", label="d_cos pool (38)"),
    ],
    fontsize=8,
)
plt.suptitle("Composition quality by pool-selection metric")
plt.tight_layout()
plt.savefig("resources/38_pool_comparison.png", dpi=150)
plt.show()

print("d_cos-pool minus expr-pool (paired by method and n):")
for n in (2, 3):
    for method in order:
        d = summary[
            (summary["pool"] == "dcos")
            & (summary["method"] == method)
            & (summary["n"] == n)
        ]
        e = summary[
            (summary["pool"] == "expr")
            & (summary["method"] == method)
            & (summary["n"] == n)
        ]
        print(
            f"  n={n} {method:9s} joint_cos {d['joint_cos'].iloc[0] - e['joint_cos'].iloc[0]:+.4f}"
            f"   all_pos_cos {d['all_positive_cos'].iloc[0] - e['all_positive_cos'].iloc[0]:+.2f}"
        )